In [19]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


In [27]:
#initial file processing
labcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = homecomp


filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
savedir = titledpath + filedir + "Compilation with delta\\2025deltagcollection\\"
saveosardir = titledpath + filedir + "Compilation with delta\\2025fallingtoosarcomp\\"
saveappendixdir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\Compilation with delta\\appendixfile\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"

In [28]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

# for file_no in os.listdir(openPath): 
#     if respondercsv in file_no and "w1118" not in file_no :   
#         f = os.path.join(openPath, file_no)
#         dfe=pd.read_csv(f)
#         exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
#         driver = file_no.split(" ")[0]
#         lstnew.append(driver)
# #lst = lstnew.copy()
# lst = [x for x in lstnew if x not in ['Th-Gal4', 'R58', "R76B09", "VT999036"]]

#processing ONLY specific names
lst = ['SS80974']
print(lst)

['SS80974']


## Regular file generation

In [29]:
for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.timerule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.timerule(NLCLIMB.generation(wtdf, wt))
       
    #processing before dabest application 
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True)
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_pp = NLMATH.bheight(NLMATH.pauseheight(dfexpt), NLMATH.pauseheight(dfwt)).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_sim = pd.concat([NLMATH.straightnessindexmeter(dfexpt, "Expt"), NLMATH.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLMATH.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLMATH.pausecomp(dfexpt, driver)
    alltgtmeandf_bout = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Bouts"), NLMATH.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Bouts"), NLMATH.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
            
    #___________________________________________#    
    # meandiff plots -- you run mean_diff instead of delta_g because since all the binary data is at the same dimension, no standardization is required and empirical delta delta is sufficient
    #dff2_prop = NLMATH.deltaversion_meandiff(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLMATH.deltaversion_meandiff(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0

    #deltag plots
    dfs2 = NLMATH.deltaversion_deltag(df_sp, "Velocity", "speed")
    dfh2 = NLMATH.deltaversion_deltag(df_h, "Y", "height")
    dfbs2 = NLMATH.deltaversion_deltag(df_bsp, "BSpeed", "bspeed")
    dfpp2 = NLMATH.deltaversion_deltag(df_pp, "Height", "pausepos")
    dfmv2 = NLMATH.deltaversion_deltag(df_maxv, "maxvelocity", "maxvelocity")
    dfsim2 = NLMATH.deltaversion_deltag(df_sim, "averagestraightnessindex", "straightindex")
    #pause and bouts
    dfmb2 = NLMATH.deltaversion_deltag(alltgtmeandf_bout, "Bouts", "meanbout")     
    dfnb2 = NLMATH.deltaversion_deltag(alltgtnumberdf_bout, "Bouts", "bout")
    
    #singledelta processing
    lsr_bsp = NLMATH.log2metric(df_bsp, "BSpeed")
    lsr_sp = NLMATH.log2metric(df_sp, 'Velocity')    
    
    #new index and ratio metrics
    bout_index_nb = NLMATH.boutindex(alltgtnumberdf_bout, "Bouts") 
    ratio_mb = NLMATH.simplemetricratio(alltgtmeandf_bout, "Bouts") 
    ratio_maxv = NLMATH.simplemetricratio(df_maxv, "maxvelocity")
    

    df_lsrbsp = NLMATH.singledelta(lsr_bsp, "log2 BSpeed", "log2bspeed")
    df_lsrsp = NLMATH.singledelta(lsr_sp, "log2 Velocity", "log2speed")
    df_boutindex_nb = NLMATH.singledelta(bout_index_nb, "Bouts", "boutnumber_index")  # Renamed from ratio to index
    df_ratio_mb = NLMATH.singledelta(ratio_mb, "Bouts", "boutduration_ratio")
    df_ratio_maxv = NLMATH.singledelta(ratio_maxv, "maxvelocity", "maxvelocity_ratio")
    
    #final df and saving into excel
    dftotal = pd.concat([dff2_number, dfs2, dfh2, dfbs2, dfpp2, dfmv2, dfsim2, dfmb2, dfnb2, df_lsrbsp, df_lsrsp, df_boutindex_nb, df_ratio_mb, df_ratio_maxv], axis = 1)
    dftotal['MBON'] = n
    dftotal.set_index("MBON", inplace = True)
    dftotal.to_csv(savedir + n + " x " + responder + "_deltag_allstats.csv")
    #dftotal.to_csv(savedir + n + " x " + responder + "_deltag_allstats.csv")
print("Done!")        

SS80974
Done!


## Thesis excel file generation

In [30]:
from dabest._stats_tools.confint_1group import summary_ci_1group

def thesis_meandiff_delta2(df, metric, df_naming, driver, responder):
    """
    Process delta2 data using mean_diff.
    For: df_f (falling data)
    Returns 4 rows: Control; Light off, Control; Light on, Test; Light off, Test; Light on
    """
    try:
        # Filter out Recovery data and null values
        df6 = df[(df['ExperimentState'] != "Recovery")]
        name = []
        if any(df6[metric].isnull()):
            name = df6[df6[metric].isnull()]['index'].tolist()
        df_clean = df6[~df6['index'].isin(name)]
        
        # Load dabest for delta2 mean_diff
        db = dabest.load(
            data=df_clean, 
            x=["ExperimentState", "Type"], 
            y=metric,  
            delta2=True, 
            experiment="Type",
            experiment_label=['WT', 'Expt'], 
            x1_level=["Dark", "Full"], 
            paired="baseline", 
            id_col="index"
        )
        
        results = db.mean_diff.results
        delta_delta_results = db.mean_diff.delta_delta.results
        
        # Check if results are valid
        if len(results) == 0:
            return pd.DataFrame()
        
        # Determine which index is WT vs Expt
        if 'WT' in str(results.control.iloc[0]):
            wt_idx = 0
            expt_idx = 1
        else:
            wt_idx = 1
            expt_idx = 0
        
        # Get plot data for mean calculations
        plot_data = db._plot_data
        xvar = db._xvar
        yvar = db._yvar
        
        # Define the 4 groups
        groups = ["Dark WT", "Full WT", "Dark Expt", "Full Expt"]
        group_labels = ["Control; Light off", "Control; Light on", "Test; Light off", "Test; Light on"]
        
        # Genotype strings
        genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
        genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
        genotypes = [genotype_control, genotype_control, genotype_test, genotype_test]
        
        rows = []
        for i, (group, label, genotype) in enumerate(zip(groups, group_labels, genotypes)):
            # Get group data for mean calculation
            group_data = plot_data[plot_data[xvar] == group][yvar].values
            
            if len(group_data) == 0:
                continue
            
            # Calculate mean and CI - handle zero-variance data (e.g., all zeros)
            if np.std(group_data) == 0:
                # All values are identical - BCA will fail, use simplified stats
                group_stats = {
                    'summary': np.mean(group_data),
                    'bca_ci_low': np.mean(group_data),
                    'bca_ci_high': np.mean(group_data)
                }
            else:
                group_stats = summary_ci_1group(
                    x=group_data,
                    func=np.mean,
                    resamples=5000,
                    alpha=0.05
                )
            
            mean_val = round(group_stats['summary'], 2)
            mean_ci_low = round(group_stats['bca_ci_low'], 2)
            mean_ci_high = round(group_stats['bca_ci_high'], 2)
            sample_size = len(group_data)
            
            # Effect Size - only for "Light on" rows (Full)
            if "Full" in group:
                if "WT" in group:
                    es_val = round(results.difference.iloc[wt_idx], 2)
                    es_ci_low = round(results.bca_low.iloc[wt_idx], 2)
                    es_ci_high = round(results.bca_high.iloc[wt_idx], 2)
                else:
                    es_val = round(results.difference.iloc[expt_idx], 2)
                    es_ci_low = round(results.bca_low.iloc[expt_idx], 2)
                    es_ci_high = round(results.bca_high.iloc[expt_idx], 2)
                delta_object = "Delta-Delta"
            else:
                es_val = " "
                es_ci_low = " "
                es_ci_high = " "
                delta_object = " "
            
            # Delta-Delta - only for Test; Light on
            if group == "Full Expt":
                dd_val = round(delta_delta_results.difference.iloc[0], 2)
                dd_ci_low = round(delta_delta_results.bca_low.iloc[0], 2)
                dd_ci_high = round(delta_delta_results.bca_high.iloc[0], 2)
            else:
                dd_val = " "
                dd_ci_low = " "
                dd_ci_high = " "
            
            rows.append({
                "MBON": driver,
                "Group": label,
                "Genotype": genotype,
                "Sample Size": sample_size,
                "Mean": mean_val,
                "Mean_CI_low": mean_ci_low,
                "Mean_CI_high": mean_ci_high,
                "Effect Size": es_val,
                "Effect Size_CI_low": es_ci_low,
                "Effect Size_CI_high": es_ci_high,
                "Delta Object": delta_object,
                "Delta-Delta/Delta-g": dd_val,
                "Delta-Delta/Delta-g_CI_low": dd_ci_low,
                "Delta-Delta/Delta-g_CI_high": dd_ci_high,
                "Metric": df_naming
            })
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        print(f"  Error in thesis_meandiff_delta2 for {driver} - {df_naming}: {e}")
        return pd.DataFrame()


def thesis_hedgesg_delta2(df, metric, df_naming, driver, responder):
    """
    Process delta2 data using hedges_g.
    For: df_sp, df_h, df_bsp, df_pp, df_maxv, df_sim, alltgtmeandf_bout, alltgtnumberdf_bout
    Returns 4 rows: Control; Light off, Control; Light on, Test; Light off, Test; Light on
    """
    try:
        # Filter out Recovery data and null values
        df6 = df[(df['ExperimentState'] != "Recovery")]
        name = []
        if any(df6[metric].isnull()):
            name = df6[df6[metric].isnull()]['index'].tolist()
        df_clean = df6[~df6['index'].isin(name)]
        
        # Load dabest for delta2 hedges_g
        db = dabest.load(
            data=df_clean, 
            x=["ExperimentState", "Type"], 
            y=metric,  
            delta2=True, 
            experiment="Type",
            experiment_label=['WT', 'Expt'], 
            x1_level=["Dark", "Full"], 
            paired="baseline", 
            id_col="index"
        )
        
        results = db.hedges_g.results
        delta_delta_results = db.hedges_g.delta_delta.results
        
        # Check if results are valid
        if len(results) == 0:
            return pd.DataFrame()
        
        # Determine which index is WT vs Expt
        if 'WT' in str(results.control.iloc[0]):
            wt_idx = 0
            expt_idx = 1
        else:
            wt_idx = 1
            expt_idx = 0
        
        # Get plot data for mean calculations
        plot_data = db._plot_data
        xvar = db._xvar
        yvar = db._yvar
        
        # Define the 4 groups
        groups = ["Dark WT", "Full WT", "Dark Expt", "Full Expt"]
        group_labels = ["Control; Light off", "Control; Light on", "Test; Light off", "Test; Light on"]
        
        # Genotype strings
        genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
        genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
        genotypes = [genotype_control, genotype_control, genotype_test, genotype_test]
        
        rows = []
        for i, (group, label, genotype) in enumerate(zip(groups, group_labels, genotypes)):
            # Get group data for mean calculation
            group_data = plot_data[plot_data[xvar] == group][yvar].values
            
            if len(group_data) == 0:
                continue
            
            # Calculate mean and CI
            group_stats = summary_ci_1group(
                x=group_data,
                func=np.mean,
                resamples=5000,
                alpha=0.05
            )
            
            mean_val = round(group_stats['summary'], 2)
            mean_ci_low = round(group_stats['bca_ci_low'], 2)
            mean_ci_high = round(group_stats['bca_ci_high'], 2)
            sample_size = len(group_data)
            
            # Effect Size - only for "Light on" rows (Full)
            if "Full" in group:
                if "WT" in group:
                    es_val = round(results.difference.iloc[wt_idx], 2)
                    es_ci_low = round(results.bca_low.iloc[wt_idx], 2)
                    es_ci_high = round(results.bca_high.iloc[wt_idx], 2)
                else:
                    es_val = round(results.difference.iloc[expt_idx], 2)
                    es_ci_low = round(results.bca_low.iloc[expt_idx], 2)
                    es_ci_high = round(results.bca_high.iloc[expt_idx], 2)
                delta_object = "Delta-g"
            else:
                es_val = " "
                es_ci_low = " "
                es_ci_high = " "
                delta_object = " "
            
            # Delta-g - only for Test; Light on
            if group == "Full Expt":
                dd_val = round(delta_delta_results.difference.iloc[0], 2)
                dd_ci_low = round(delta_delta_results.bca_low.iloc[0], 2)
                dd_ci_high = round(delta_delta_results.bca_high.iloc[0], 2)
            else:
                dd_val = " "
                dd_ci_low = " "
                dd_ci_high = " "
            
            rows.append({
                "MBON": driver,
                "Group": label,
                "Genotype": genotype,
                "Sample Size": sample_size,
                "Mean": mean_val,
                "Mean_CI_low": mean_ci_low,
                "Mean_CI_high": mean_ci_high,
                "Effect Size": es_val,
                "Effect Size_CI_low": es_ci_low,
                "Effect Size_CI_high": es_ci_high,
                "Delta Object": delta_object,
                "Delta-Delta/Delta-g": dd_val,
                "Delta-Delta/Delta-g_CI_low": dd_ci_low,
                "Delta-Delta/Delta-g_CI_high": dd_ci_high,
                "Metric": df_naming
            })
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        print(f"  Error in thesis_hedgesg_delta2 for {driver} - {df_naming}: {e}")
        return pd.DataFrame()


def thesis_hedgesg_single(df, metric, df_naming, driver, responder):
    """
    Process single delta data using hedges_g (no delta-delta/delta-g).
    For: lsr_bsp, lsr_sp, bout_index_nb, ratio_mb, ratio_maxv
    Returns 2 rows: Control, Test
    """
    try:
        # Load dabest for single hedges_g comparison
        db = dabest.load(df, idx=("WT", "Expt"), y=metric, x='Type')
        
        results = db.hedges_g.results
        
        # Check if results are valid
        if len(results) == 0:
            return pd.DataFrame()
        
        # Get plot data for mean calculations
        plot_data = db._plot_data
        xvar = db._xvar
        yvar = db._yvar
        
        # Define the 2 groups
        groups = ["WT", "Expt"]
        group_labels = ["Control", "Test"]
        
        # Genotype strings
        genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
        genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
        genotypes = [genotype_control, genotype_test]
        
        rows = []
        for i, (group, label, genotype) in enumerate(zip(groups, group_labels, genotypes)):
            # Get group data for mean calculation
            group_data = plot_data[plot_data[xvar] == group][yvar].values
            
            if len(group_data) == 0:
                continue
            
            # Calculate mean and CI
            group_stats = summary_ci_1group(
                x=group_data,
                func=np.mean,
                resamples=5000,
                alpha=0.05
            )
            
            mean_val = round(group_stats['summary'], 2)
            mean_ci_low = round(group_stats['bca_ci_low'], 2)
            mean_ci_high = round(group_stats['bca_ci_high'], 2)
            sample_size = len(group_data)
            
            # Effect Size - only for Test row
            if group == "Expt":
                es_val = round(results.difference.iloc[0], 2)
                es_ci_low = round(results.bca_low.iloc[0], 2)
                es_ci_high = round(results.bca_high.iloc[0], 2)
                delta_object = "Hedges' g"
            else:
                es_val = " "
                es_ci_low = " "
                es_ci_high = " "
                delta_object = " "
            
            rows.append({
                "MBON": driver,
                "Group": label,
                "Genotype": genotype,
                "Sample Size": sample_size,
                "Mean": mean_val,
                "Mean_CI_low": mean_ci_low,
                "Mean_CI_high": mean_ci_high,
                "Effect Size": es_val,
                "Effect Size_CI_low": es_ci_low,
                "Effect Size_CI_high": es_ci_high,
                "Delta Object": delta_object,
                "Delta-Delta/Delta-g": " ",
                "Delta-Delta/Delta-g_CI_low": " ",
                "Delta-Delta/Delta-g_CI_high": " ",
                "Metric": df_naming
            })
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        print(f"  Error in thesis_hedgesg_single for {driver} - {df_naming}: {e}")
        return pd.DataFrame()

In [31]:
for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.timerule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.timerule(NLCLIMB.generation(wtdf, wt))
       
    #processing before dabest application 
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True)
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_pp = NLMATH.bheight(NLMATH.pauseheight(dfexpt), NLMATH.pauseheight(dfwt)).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_sim = pd.concat([NLMATH.straightnessindexmeter(dfexpt, "Expt"), NLMATH.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLMATH.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLMATH.pausecomp(dfexpt, driver)
    alltgtmeandf_bout = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Bouts"), NLMATH.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Bouts"), NLMATH.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    
    #singledelta processing
    lsr_bsp = NLMATH.log2metric(df_bsp, "BSpeed")
    lsr_sp = NLMATH.log2metric(df_sp, 'Velocity')    
    
    #new index and ratio metrics
    bout_index_nb = NLMATH.boutindex(alltgtnumberdf_bout, "Bouts") 
    ratio_mb = NLMATH.simplemetricratio(alltgtmeandf_bout, "Bouts") 
    ratio_maxv = NLMATH.simplemetricratio(df_maxv, "maxvelocity")

    # Thesis DataFrame creation using the new functions
    # Collect all DataFrames, filtering out empty ones
    thesis_dfs = [
        # Bracket 1 - mean_diff + delta-delta
        thesis_meandiff_delta2(df_f, "Fall", "fallnumber", driver, responder),
        
        # Bracket 2 - hedges_g + delta g
        thesis_hedgesg_delta2(df_sp, "Velocity", "speed", driver, responder),
        thesis_hedgesg_delta2(df_h, "Y", "height", driver, responder),
        thesis_hedgesg_delta2(df_bsp, "BSpeed", "bspeed", driver, responder),
        thesis_hedgesg_delta2(df_pp, "Height", "pausepos", driver, responder),
        thesis_hedgesg_delta2(df_maxv, "maxvelocity", "maxvelocity", driver, responder),
        thesis_hedgesg_delta2(df_sim, "averagestraightnessindex", "straightindex", driver, responder),
        thesis_hedgesg_delta2(alltgtmeandf_bout, "Bouts", "meanbout", driver, responder),
        thesis_hedgesg_delta2(alltgtnumberdf_bout, "Bouts", "boutnumber", driver, responder),
        
        # Bracket 3 - hedges_g only (no delta)
        thesis_hedgesg_single(lsr_sp, "log2 Velocity", "log2speed", driver, responder),
        thesis_hedgesg_single(lsr_bsp, "log2 BSpeed", "log2bspeed", driver, responder),
        thesis_hedgesg_single(bout_index_nb, "Bouts", "boutnumber_index", driver, responder),
        thesis_hedgesg_single(ratio_mb, "Bouts", "boutduration_ratio", driver, responder),
        thesis_hedgesg_single(ratio_maxv, "maxvelocity", "maxvelocity_ratio", driver, responder),
    ]
    
    # Filter out empty DataFrames before concatenating
    thesis_dfs = [df for df in thesis_dfs if len(df) > 0]
    
    if len(thesis_dfs) > 0:
        df_thesis_final = pd.concat(thesis_dfs, ignore_index=True)
        # Save to CSV
        df_thesis_final.to_csv("D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\Compilation with delta\\2025thesisstats\\" + n + " x " + responder + "_thesis_stats.csv", index=False)
    else:
        print(f"  Warning: No valid data for {driver}")
    
print("Done!")

SS80974
Done!
